## 📑 Core Modifications Production

#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AQAhHg)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
import boto3
import os

#takes the values d for dev, {env_prefix} for test and p for prod depending in which workspace the init variables script is running
# env = os.environ"S3PREFIX" 

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

som_date = spark.sql(
f'''
SELECT 
     MAX(cal.CalendarDate) CalendarDate
FROM `{env_prefix}-cdh`.reference.ref_calendar AS cal
WHERE cal.IsBusinessDay = 1
AND cal.CalendarDate <= LAST_DAY(DATE_ADD(MONTH, -1, {run_date!r}))
''').collect()[0][0]

soq_date = spark.sql(
f'''
SELECT 
     MAX(cal.CalendarDate) CalendarDate
FROM `{env_prefix}-cdh`.reference.ref_calendar AS cal
WHERE cal.IsBusinessDay = 1
AND cal.CalendarDate <= date_add(QUARTER, -1, DATE {run_date!r})
''').collect()[0][0]



###### Schema of the table

In [0]:
table_name = 'Core_ModificationsProduction'

schema = (
"BusinessDate date," 
"CDI string,"
"PartyTaxRegistrationNumber string,"
"CustomerName string,"
"L3869Status string,"
"AgreementKey string,"
"ExposureKey string,"
"AssetSegment string,"
"TreatmentBranch string,"
"Branch string,"
"SoMBalance decimal(19,2),"
"SoMBucket int,"
"SoMDenouncedStatus string,"
"RiskBucket int,"
"EBAStatus string,"
"AllFinalEBAStatus string,"
"PENPEAccount string,"
"PE0PE1_3NPEAccount string,"
"PENPEDebtor string,"
"PENPEHubDebtor string,"
"CourseOfAction string,"
"CustomerRetailCategory string,"
"AssignmentChannel string,"
"AssignmentChannelDetailed string,"
"AssignmentAgent string,"
"AssignmentAgentAM string,"
"AccountSegment string,"
"AssignmentSegmentDetailed string,"
"FinalSavedFlag string,"
"FinalSavedReason string,"
"ImplementationDate date,"
"ProductType string,"
"Forbearance string,"
"LowInstallmentDate date,"
"TargetDate date,"
"InflowMonth string,"
"ActivePipelineSoQ int,"
"ActivePipelineSoM int,"
"NewPipelineWithinQuarter int,"
"NewPipelineWithinMonth int,"
"ImplementedWithinQuarter int,"
"ImplementedWithinMonth int,"
"ImplementedWithHaircutWithinMonth int,"
"ApplicationNumber string,"
"ApplicationInsertDate date,"
"StepCategory string,"
"ApplicationCurrentStep string,"
"ApplicationCurrentStepInsertDate date,"
"FirstDateAtValidation date,"
"ApplicationChannelCategory string,"
"ApplicationStatus string,"
"ApplicationApproval string,"
"ApplicationApprovalReason string,"
"NPLSolution int,"
"DEK int,"
"ModificationType string,"
"HaircutAmount decimal(19,2),"
"ReturnReason string,"
"ImplementationApplicationNumber string,"
"ApplicationLevelAutoCredit int,"
"ReevaluationTimes int,"
"ApplicationRegistratorAM string,"
"ApplicationRegistratorName string,"
"ApplicationBranch string,"
"FollowBranch string,"
"AutomaticPrinting string,"
"PrintException string,"
"IndicatorOfAutomatedPrintingAtApplicationLevel int,"
"IndicatorPreapprovedAtApplicationLevel int,"
"ApplicationAutocredit string,"
"InitialSignature string,"
"ApprovalDifferentTerms string,"
"ApplicationApprovalWithDifferentTerms string,"
"ApplicationSignatureRequired string,"
"SignedByBankDate date,"
"L4605Flag string,"
"Covid19Flag string,"
"DiscretionOfApproval string,"
"DNCcategory string,"
"NewPipelineWithinMonth_IRT int,"
"ImplementedWithinMonth_IRT int,"
"ActivityTypeKey_IRT string,"
"ActivityName_IRT string,"
"ActivityDate_IRT date,"
"ActivityUser_IRT string,"
"ActivityUserId_IRT string,"
"FinalAssignmentChannel string,"
"FinalAssignmentTeam string,"
"FinalAssignmentAgent string,"
"FinalAssignmentAgentAM string,"
"AttributionChannel string,"
"AttributionTeam string,"
"AttributionAgent string,"
"AttributionAgentAM string,"
"PaymentAttributionChannel string,"
"PaymentAttributionTeam string,"
"PaymentAttributionAgent string,"
"PaymentAttributionAgentAM string,"
"PaymentAttributionAmount decimal(19,2),"
"ExternalFileSource int,"
"InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### Create wanted Delta Tables with selected fields & Temporary Views for each delta table

In [0]:
Qualco = spark.sql(
f'''
select ACCOUNT_NUMBER ,LAST_PAY_AMOUNT ,LAST_PAY_DATE ,OBJECTIVE_STATUS ,OBJECTIVE_STATUS_DATE ,DEBT_LPS ,DEBT_LPS_DATE ,BRANCH_CODE ,BRANCH_DESC 
from `{env_prefix}-cdh-delivery`.repoint.cl_fullportfoliocardsprm
where EXPRT_MONTH = LEFT(REPLACE(CAST({som_date!r} AS STRING), '-', ''), 6)
and BusinessDate = (select max(BusinessDate) from `{env_prefix}-cdh-delivery`.repoint.cl_fullportfoliocardsprm where EXPRT_MONTH = LEFT(REPLACE(CAST({som_date!r} AS STRING), '-', ''), 6))
union
select ACCOUNT_NUMBER ,LAST_PAY_AMOUNT ,LAST_PAY_DATE ,OBJECTIVE_STATUS ,OBJECTIVE_STATUS_DATE ,DEBT_LPS ,DEBT_LPS_DATE ,BRANCH_CODE ,BRANCH_DESC 
from `{env_prefix}-cdh-delivery`.repoint.cl_fullportfolioloansprm
where EXPRT_MONTH = LEFT(REPLACE(CAST({som_date!r} AS STRING), '-', ''), 6)
and BusinessDate = (select max(BusinessDate) from `{env_prefix}-cdh-delivery`.repoint.cl_fullportfoliocardsprm where EXPRT_MONTH = LEFT(REPLACE(CAST({som_date!r} AS STRING), '-', ''), 6))
union
select ACCOUNT_NUMBER ,LAST_PAY_AMOUNT ,LAST_PAY_DATE ,OBJECTIVE_STATUS ,OBJECTIVE_STATUS_DATE ,DEBT_LPS ,DEBT_LPS_DATE ,BRANCH_CODE ,BRANCH_DESC
from `{env_prefix}-cdh-delivery`.repoint.ml_fullportfolioprm
where EXPRT_MONTH = LEFT(REPLACE(CAST({som_date!r} AS STRING), '-', ''), 6)
and BusinessDate = (select max(BusinessDate) from `{env_prefix}-cdh-delivery`.repoint.cl_fullportfoliocardsprm where EXPRT_MONTH = LEFT(REPLACE(CAST({som_date!r} AS STRING), '-', ''), 6))
union
select ACCOUNT_NUMBER ,LAST_PAY_AMOUNT ,LAST_PAY_DATE ,OBJECTIVE_STATUS ,OBJECTIVE_STATUS_DATE ,DEBT_LPS ,DEBT_LPS_DATE ,BRANCH_CODE ,BRANCH_DESC
from `{env_prefix}-cdh-delivery`.repoint.sb_fullportfolioprm
where EXPRT_MONTH = LEFT(REPLACE(CAST({som_date!r} AS STRING), '-', ''), 6)
and BusinessDate = (select max(BusinessDate) from `{env_prefix}-cdh-delivery`.repoint.cl_fullportfoliocardsprm where EXPRT_MONTH = LEFT(REPLACE(CAST({som_date!r} AS STRING), '-', ''), 6))
'''
)
Qualco.createOrReplaceTempView("Qualco")

##### Assignment Data Mapping

In [0]:
AssignmentDataMapping = spark.sql(
f'''
SELECT
				m.CDI
			,	m.Entity
			,	m.PartyTaxRegistrationNumber
			,	m.AgreementKey
			,	m.ExposureKey
			,	m.AssetSegment                      
			,	m.Treatmentbranch                   
			,	m.SoMBalanceAmountEUR               
			,	m.SoMSourceBucket
   			,	m.SoMDenouncedStatus  
			,	m.RiskBucket
			,	m.EBAStatus 
			,	m.CourseOfAction                   
			,	m.AssignmentChannel 
			,	m.AssignmentTeam                   
			,	m.AssignmentAgent                  
			,	m.AssignmentSegmentDetailed
			,	LEFT(m.AssignmentTeam,4) AS AssignmentHub
			,	m.AttributionChannel
   			,	m.AttributionTeam
			,	m.AttributionAgent
			,	m.AttributionAgentAM
			,	m.Event
			,	m.PaymentAmount
FROM `{env_prefix}-cdh-delivery`.repoint.core_assignmentdatamapping m
WHERE m.BusinessDate = {run_date!r}
'''
)
AssignmentDataMapping.createOrReplaceTempView("AssignmentDataMapping")

##### dparty

In [0]:
dparty = spark.sql(
f'''
SELECT
	    p.CDI
        ,   concat(p.FirstNameGR, ' ',p.LastNameGR) FullName
FROM `{env_prefix}-cdh`.derivative.dparty p
WHERE p.BusinessDate = {som_date!r}
AND p.SourceSystem = 'qca'
'''
)
dparty.createOrReplaceTempView("dparty")

##### dunderlaw

In [0]:
dunderlaw = spark.sql(
f'''
SELECT
	        ul.CDI
        ,   ul.CalcLawStatus
FROM `{env_prefix}-cdh`.derivative.dunderlaw ul
WHERE ul.BusinessDate = {run_date!r}
AND ul.UnderLawType = 'N3869'
AND ul.CalcLawStatus in ('Pending Court Decision','Positive Court Decision')
'''
)
dunderlaw.createOrReplaceTempView("dunderlaw")

##### SavedInfo

In [0]:
SavedInfo = spark.sql(
f'''
SELECT * 
FROM (
SELECT DISTINCT
                BusinessDate
            ,	Account AS AccountNumber											 
			,   SavedFlag AS FinalSaved		   
			,	SavedDate AS FinalSavedDate   
		    ,   SavedReason
            -- ,   InflowTargetDateCalc AS TargetDate
            -- ,   InflowTargetDate
            ,   rank() over (partition by Account order by InflowTargetDateCalc desc) as rank
FROM `{env_prefix}-cdh`.derivative.dportfoliotoolcore
-- FROM `{env_prefix}-cdh-delivery`.repoint.dportfoliotoolcore_uat_27
WHERE BusinessDate = {run_date!r}
AND RelationType = 'Primary Owner') a 
WHERE a.rank = 1
'''
)
SavedInfo.createOrReplaceTempView("SavedInfo")

In [0]:
extreport_unified_cl_soq = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_cl
WHERE BusinessDate = {soq_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_cl WHERE BusinessDate = {soq_date!r})
''')
extreport_unified_cl_soq.createOrReplaceTempView("extreport_unified_cl_soq")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_cl_soq.count() == 0:
    print(f"The required external file extreport_unified_cl dataset for date {soq_date!r} is empty")

extreport_unified_ml_soq = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_ml
WHERE BusinessDate = {soq_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_ml WHERE BusinessDate = {soq_date!r})
''')
extreport_unified_ml_soq.createOrReplaceTempView("extreport_unified_ml_soq")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_ml_soq.count() == 0:
    print(f"The required external file extreport_unified_ml dataset for date {soq_date!r} is empty")

extreport_unified_sb_soq = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_sb
WHERE BusinessDate = {soq_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_sb WHERE BusinessDate = {soq_date!r})
''')
extreport_unified_sb_soq.createOrReplaceTempView("extreport_unified_sb_soq")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_sb_soq.count() == 0:
    print(f"The required external file extreport_unified_sb dataset for date {soq_date!r} is empty")


extreport_unified_cl_som = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_cl
WHERE BusinessDate = {som_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_cl WHERE BusinessDate = {som_date!r})
''')
extreport_unified_cl_som.createOrReplaceTempView("extreport_unified_cl_som")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_cl_som.count() == 0:
    print(f"The required external file extreport_unified_cl dataset for date {som_date!r} is empty")

extreport_unified_ml_som = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_ml
WHERE BusinessDate = {som_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_ml WHERE BusinessDate = {som_date!r})
''')
extreport_unified_ml_som.createOrReplaceTempView("extreport_unified_ml_som")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_ml_som.count() == 0:
    print(f"The required external file extreport_unified_ml dataset for date {som_date!r} is empty")

extreport_unified_sb_som = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_sb
WHERE BusinessDate = {som_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_sb WHERE BusinessDate = {som_date!r})
''')
extreport_unified_sb_som.createOrReplaceTempView("extreport_unified_sb_som")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_sb_som.count() == 0:
    print(f"The required external file extreport_unified_sb dataset for date {som_date!r} is empty")


extreport_unified_cl = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_cl
WHERE BusinessDate = {run_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_cl WHERE BusinessDate = {run_date!r})
''')
extreport_unified_cl.createOrReplaceTempView("extreport_unified_cl")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_cl.count() == 0:
    print(f"The required external file extreport_unified_cl dataset for date {run_date!r} is empty")

extreport_unified_ml = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_ml
WHERE BusinessDate = {run_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_ml WHERE BusinessDate = {run_date!r})
''')
extreport_unified_ml.createOrReplaceTempView("extreport_unified_ml")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_ml.count() == 0:
    print(f"The required external file extreport_unified_ml dataset for date {run_date!r} is empty")

extreport_unified_sb = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_sb
WHERE BusinessDate = {run_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_sb WHERE BusinessDate = {run_date!r})
''')
extreport_unified_sb.createOrReplaceTempView("extreport_unified_sb")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_sb.count() == 0:
    print(f"The required external file extreport_unified_sb dataset for date {run_date!r} is empty")



##### reportunified

In [0]:
reportunified = spark.sql(f"""
SELECT 
	BusinessDate,
    ExposureKey,
    'CL' as AssetSegment,
    ApplicationNumber as ApplicationNumber,
    CAST(CONCAT('20', left(ApplicationNumber,2), '-', SUBSTRING(ApplicationNumber,3,2), '-', SUBSTRING(ApplicationNumber,5,2)) AS DATE) AS ApplicationInsertDate,
    CommonApplicationStep as CurrentApplicationStep,
    CAST(ReceiptDateCurrentStep AS DATE) as CurrentStepDate,
    UnifiedApplicationStatus,
    CASE   when FirstDateReceiptValuation= '' then null
        -- If the value matches the format 'YYYY-MM-DD HH:mm:ss.0'
        WHEN SUBSTRING(FirstDateReceiptValuation, 5, 1) = '-' THEN cast(FirstDateReceiptValuation AS date)
        -- If the value matches the format 'DD/MM/YYYY'
        WHEN SUBSTRING(FirstDateReceiptValuation, 3, 1) = '/' THEN TO_DATE(FirstDateReceiptValuation, 'dd/MM/yyyy')
        -- For any other format, assuming it's already in the desired format
    END AS FirstDateReceiptValuation,
    NPLSolution ,
    Decision ,
    DecisionReason ,
    DEK , 
    RestructuringType as RestructuringType,
    CASE WHEN trim(RestrectedDueAmount) in('','NULL') THEN null ELSE cast(RestrectedDueAmount as decimal(19,2)) END as HaircutAmount,
    ReturnReason ,
    ImplementationApplicationNumber,
    AutoCreditAtApplicationLevel,
    RegistratorCode,
    RegistratorName,
    ApplicationActivationBranch,
    ApplicationBranch as FollowBranch,
    AutoPrintFlag,
    PrintException,
    ApplicationAutoPrintFlag,
    ApplicationPreAppovedFlag,
    ApplicationAutocredit,
    InitialSignature,
    ApprovalDifferentTerms,
    ApplicationApprovalWithDifferentTerms,
    ApplicationSignatureRequired,
    SignedByBankDate,
    L4605Flag,
    Covid19,
    Authority,
    CASE WHEN Hub in('','NULL') THEN NULL ELSE Hub END AS Hub,
    CASE WHEN Channel in('','NULL') THEN NULL ELSE Channel END AS Channel,
    ReEvaluationTimes
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_cl
WHERE BusinessDate IN ({soq_date!r}, {som_date!r}, {run_date!r})

UNION ALL
 
SELECT 
    BusinessDate,
    ExposureKey,
    'ML' as AssetSegment,
    ApplicationNumber as ApplicationNumber,
    CAST(CONCAT('20', left(ApplicationNumber,2), '-', SUBSTRING(ApplicationNumber,3,2), '-', SUBSTRING(ApplicationNumber,5,2)) AS DATE) AS ApplicationInsertDate,
    CommonApplicationStep as CurrentApplicationStep,
    CAST(CurrentStepDate AS DATE)  as CurrentStepDate,
    UnifiedApplicationStatus,
    CASE   when FirstDateReceiptValuation= '' then null
        -- If the value matches the format 'YYYY-MM-DD HH:mm:ss.0'
        WHEN SUBSTRING(FirstDateReceiptValuation, 5, 1) = '-' THEN cast(FirstDateReceiptValuation AS date)
        -- If the value matches the format 'DD/MM/YYYY'
        WHEN SUBSTRING(FirstDateReceiptValuation, 3, 1) = '/' THEN TO_DATE(FirstDateReceiptValuation, 'dd/MM/yyyy')
        -- For any other format, assuming it's already in the desired format
    END AS FirstDateReceiptValuation,
    NPLSolution ,
    Decision ,
    DecisionReason ,
    DEK , 
    RestructuringType as RestructuringType,
    CASE WHEN trim(RestrectedDueAmount) in('','NULL') THEN null ELSE cast(RestrectedDueAmount as decimal(19,2)) END as HaircutAmount,
    ReturnReason ,
    ImplementationApplicationNumber,
    AutoCreditAtApplicationLevel,
    RegistratorCode,
    RegistratorName,
    ApplicationActivationBranch,
    ApplicationBranch as FollowBranch,
    AutoPrintFlag,
    PrintException,
    ApplicationAutoPrintFlag,
    ApplicationPreAppovedFlag,
    ApplicationAutocredit,
    InitialSignature,
    ApprovalDifferentTerms,
    ApplicationApprovalWithDifferentTerms,
    ApplicationSignatureRequired,
    SignedByBankDate,
    L4605Flag,
    Covid19,
    Authority,
    CASE WHEN Hub in('','NULL') THEN NULL ELSE Hub END AS Hub,
    CASE WHEN Channel in('','NULL') THEN NULL ELSE Channel END AS Channel,
    ReEvaluationTimes
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_ml
WHERE BusinessDate IN ({soq_date!r}, {som_date!r}, {run_date!r})

UNION ALL
 
SELECT BusinessDate,
    ExposureKey,
    'SB' as AssetSegment,
    ApplicationNumber as ApplicationNumber,
    CAST(CONCAT('20', left(ApplicationNumber,2), '-', SUBSTRING(ApplicationNumber,3,2), '-', SUBSTRING(ApplicationNumber,5,2)) AS DATE) AS ApplicationInsertDate,
    Description as CurrentApplicationStep,
    CASE   when CurrentStepDate= '' then null
        -- If the value matches the format 'YYYY-MM-DD HH:mm:ss.0'
        WHEN SUBSTRING(CurrentStepDate, 5, 1) = '-' THEN cast(CurrentStepDate AS date)
        -- If the value matches the format 'DD/MM/YYYY'
        WHEN SUBSTRING(CurrentStepDate, 3, 1) = '/' THEN TO_DATE(CurrentStepDate, 'dd/MM/yyyy')
        -- For any other format, assuming it's already in the desired format
    END as CurrentStepDate,
    UnifiedApplicationStatus,
    CASE   when FirstDateReceiptValuation= '' then null
        -- If the value matches the format 'YYYY-MM-DD HH:mm:ss.0'
        WHEN SUBSTRING(FirstDateReceiptValuation, 5, 1) = '-' THEN cast(FirstDateReceiptValuation AS date)
        -- If the value matches the format 'DD/MM/YYYY'
        WHEN SUBSTRING(FirstDateReceiptValuation, 3, 1) = '/' THEN TO_DATE(FirstDateReceiptValuation, 'dd/MM/yyyy')
        -- For any other format, assuming it's already in the desired format
    END AS FirstDateReceiptValuation,
    NPLSolution ,
    Desicion as Decision,
    DecisionReason ,
    DEK , 
    ResrtucturingType,
    cast(null as decimal(19,2)) as HaircutAmount,
    ReturnReason ,
    ImplementationApplicationNumber,
    AutoCreditAtApplicationLevel,
    RegistratorCode,
    RegistratorName,
    ApplicationActivationBranch,
    ApplicationBranch as FollowBranch,
    AutoPrintFlag,
    PrintException,
    ApplicationAutoPrintFlag,
    ApplicationPreAppovedFlag,
    ApplicationAutocredit,
    InitialSignature,
    ApprovalDifferentTerms,
    ApplicationApprovalWithDifferentTerms,
    ApplicationSignatureRequired,
    SignedByBankDate,
    L4605Flag,
    Covid19,
    Authority,
    CASE WHEN NPLHUBS in('','NULL') THEN NULL ELSE NPLHUBS END AS Hub,
    CASE WHEN Channel in('','NULL') THEN NULL ELSE Channel END AS Channel,
    `CounterΒ` AS ReEvaluationTimes
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_sb
WHERE BusinessDate IN ({soq_date!r}, {som_date!r}, {run_date!r})
""")
 
reportunified.createOrReplaceTempView("reportunified")

##### attributionchannel_restructure

In [0]:
attributionchannel_restructure = spark.sql(
f'''
SELECT * 
FROM (
SELECT DISTINCT
        ExposureKey
	,	AttributionChannel
	,	AttributionTeam
	,	AttributionAgent
	,	AttributionAgentAM
	, 	rank() over (partition by ExposureKey order by AttributionChannel) Rank
FROM AssignmentDataMapping
WHERE Event IN ('Saved due to Restructure' , 'Saved due to Extension', 'Extension','Restructure','Active Application')
AND AttributionChannel IS NOT NULL) a
WHERE a.Rank = 1
'''
)
attributionchannel_restructure.createOrReplaceTempView("attributionchannel_restructure")

##### attributionchannel_payment

In [0]:
attributionchannel_payment = spark.sql(
f'''
SELECT * 
FROM (
SELECT DISTINCT
      		ExposureKey
	,		AttributionChannel
	,		AttributionTeam
 	,		AttributionAgent
	,		AttributionAgentAM
	,		SUM(PaymentAmount) AS PaymentAmount
	, 	rank() over (partition by ExposureKey order by AttributionChannel) Rank
FROM AssignmentDataMapping
WHERE Event IN ('Saved due to Payment') 
AND AttributionChannel IS NOT NULL
GROUP BY ExposureKey,	AttributionChannel, AttributionTeam, AttributionAgent, AttributionAgentAM) a
WHERE a.Rank = 1
'''
)
attributionchannel_payment.createOrReplaceTempView("attributionchannel_payment")

##### interaction

In [0]:
interaction = spark.sql(
f'''
SELECT
        ia.ExposureKey
    ,   ia.InteractionTypeKey
    ,   ia.InteractionType
    ,   ia.InteractionDate
    ,   ia.CreatedBy
    ,   cast(null as string) as CreatedById
	,   RANK() OVER(PARTITION BY ia.ExposureKey ORDER BY ia.InteractionDate DESC, ia.InteractionTypeKey ASC, ia.CreatedBy DESC ) AS RANK 
FROM `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) ia
WHERE ia.InteractionDate > '2022-12-31'
and ia.InteractionDate <= {run_date!r}
and ia.InteractionTypeKey IN ('500','536','547','578','620','486')
'''
)
interaction.createOrReplaceTempView("interaction")

##### unionfile

In [0]:
last_day_soq_date = spark.sql(f'''SELECT cast(last_day({soq_date!r}) as string)''').collect()[0][0]

unionfile = spark.sql(
f'''
SELECT 
    uf.SnapshotDate
,	uf.ExposureNumber
,	uf.EBAStatus
,	uf.ProbationStartDate
,	uf.BucketCategoryWithMateriality
,	uf.FPLUnderProbationFlag
,   uf.ForbearanceMeasuresCount
,   case when uf.LTV = '' then null else cast(uf.LTV as decimal(19,2)) end LTV
FROM `{env_prefix}-cdh`.externalfiles.extunionfile uf
INNER JOIN (SELECT SnapshotDate, max(IngestionTimestamp) AS IngestionTimestamp
            FROM `{env_prefix}-cdh`.externalfiles.extunionfile
            WHERE SnapshotDate = {last_day_soq_date!r}
            GROUP BY SnapshotDate) uf0 
            ON uf.IngestionTimestamp = uf0.IngestionTimestamp and uf.SnapshotDate = uf0.SnapshotDate
WHERE uf.SnapshotDate = {last_day_soq_date!r}
'''
)
unionfile.createOrReplaceTempView("unionfile")

# Check if the dataset is empty and print a message accordingly
if unionfile.count() == 0:
    print(f"The required external file extunionfile dataset for date {last_day_soq_date!r} is empty")

##### pmmodifications

In [0]:
pmmodifications = spark.sql(
f'''
SELECT DISTINCT
		    M.ExposureKey
    	,	M.ImplementationDate
		,	M.Segment
		,	M.ModificationProduct
    	,   M.NPEFinalizedInflow
		,   M.EBAIndicator
		,	M.BusinessDate
FROM `{env_prefix}-cdh-delivery`.repoint.core_pmmodifications M
WHERE BusinessDate = {run_date!r}
'''
)
pmmodifications.createOrReplaceTempView("pmmodifications")

##### AssignmentDataMapping_v1

In [0]:
AssignmentDataMapping_v1 = spark.sql(
f'''
SELECT DISTINCT
			m.CDI
		,	m.Entity
		,	m.PartyTaxRegistrationNumber
		,	m.AgreementKey
		,	m.ExposureKey
		,	m.AssetSegment                      
		,	m.Treatmentbranch                   
		,	m.SoMBalanceAmountEUR               
		,	m.SoMSourceBucket 		               	
		,	m.SoMDenouncedStatus  
		,	m.RiskBucket	
		,	m.EBAStatus 
		,	m.CourseOfAction                   
		,	m.AssignmentChannel                
		,	m.AssignmentTeam                   
		,	m.AssignmentAgent                  
		,	m.AssignmentSegmentDetailed
		,	LEFT(IFNULL(attr_restr.AttributionTeam, m.AssignmentTeam),4)  AssignmentHub
        ,	CASE WHEN m.EBAStatus IN ('NPL','FNPL')		THEN 'NPE'
				 WHEN m.EBAStatus IN ('PL','FPL','CFPL')  THEN 'PE'
			END AS PENPEAccount
		,	CASE WHEN MONTH({run_date!r}) % 3 = 1 AND m.EBAStatus IN ('NPL','FNPL')										THEN 'NPE' 
			 	 WHEN MONTH({run_date!r}) % 3 = 1 AND m.EBAStatus IN ('PL','FPL','CFPL') AND m.RiskBucket = 0			THEN 'PE 0'  
			 	 WHEN MONTH({run_date!r}) % 3 = 1 AND m.EBAStatus IN ('PL','FPL','CFPL') AND m.RiskBucket IN (1,2,3)	THEN 'PE 1-3'
			 	 WHEN MONTH({run_date!r}) % 3 = 2 AND m.EBAStatus IN ('NPL','FNPL')										THEN 'NPE' 
			 	 WHEN MONTH({run_date!r}) % 3 = 2 AND m.EBAStatus IN ('PL','FPL','CFPL') AND m.RiskBucket IN (0,1)	THEN 'PE 0'  
			 	 WHEN MONTH({run_date!r}) % 3 = 2 AND m.EBAStatus IN ('PL','FPL','CFPL') AND m.RiskBucket IN (2,3)	THEN 'PE 1-3'
				 WHEN MONTH({run_date!r}) % 3 = 0 AND m.EBAStatus IN ('NPL','FNPL')										THEN 'NPE' 
			 	 WHEN MONTH({run_date!r}) % 3 = 0 AND m.EBAStatus IN ('PL','FPL','CFPL') AND m.RiskBucket IN (0,1,2)	THEN 'PE 0'  
			 	 WHEN MONTH({run_date!r}) % 3 = 0 AND m.EBAStatus IN ('PL','FPL','CFPL') AND m.RiskBucket = 3			THEN 'PE 1-3' 
			END AS PE0PE1_3NPEAccount
		,	u.LTV
		,	CASE WHEN u.LTV <  1.2 THEN 'LTV < 120%'
				 WHEN u.LTV >= 1.2 THEN 'LTV >= 120%'
				 ELSE 'Unsecured'
			END AS SecuredFlag
		,	q.LAST_PAY_AMOUNT
		,	q.LAST_PAY_DATE
		,	q.OBJECTIVE_STATUS
		,	q.OBJECTIVE_STATUS_DATE
		,	q.DEBT_LPS
		,	q.BRANCH_CODE
		,	q.BRANCH_DESC	
	    ,	CASE WHEN attr_restr.AttributionChannel IS NOT NULL THEN attr_restr.AttributionChannel 
     			 ELSE m.AssignmentChannel END AS FinalAssignmentChannel
        ,	CASE WHEN attr_restr.AttributionChannel IS NOT NULL THEN attr_restr.AttributionTeam 
     			 ELSE m.AssignmentTeam END AS FinalAssignmentTeam
        ,	CASE WHEN attr_restr.AttributionChannel IS NOT NULL THEN attr_restr.AttributionAgent 
     			 ELSE m.AssignmentAgent END AS FinalAssignmentAgent
        ,	CASE WHEN attr_restr.AttributionChannel IS NOT NULL THEN attr_restr.AttributionAgentAM 
     			 ELSE NULL END AS FinalAssignmentAgentAM
      	,	attr_restr.AttributionChannel  AS AttributionChannel
	    ,	attr_restr.AttributionTeam  AS AttributionTeam
	    ,	attr_restr.AttributionAgent AS AttributionAgent 
	    ,	attr_restr.AttributionAgentAM AS AttributionAgentAM
		,	attr_pay.AttributionChannel	AS PaymentAttributionChannel
		,	attr_pay.AttributionTeam AS PaymentAttributionTeam
		,	attr_pay.AttributionAgent AS PaymentAttributionAgent
		,	attr_pay.AttributionAgentAM AS PaymentAttributionAgentAM
		,	attr_pay.PaymentAmount AS PaymentAttributionAmount
FROM AssignmentDataMapping m
INNER JOIN pmmodifications pm ON pm.ExposureKey = m.ExposureKey AND pm.BusinessDate = {run_date!r}
LEFT JOIN Qualco q ON q.ACCOUNT_NUMBER = m.ExposureKey
LEFT JOIN UnionFile u ON u.ExposureNumber = m.ExposureKey
LEFT JOIN attributionchannel_restructure attr_restr ON attr_restr.ExposureKey = m.ExposureKey 
LEFT JOIN attributionchannel_payment attr_pay ON attr_pay.ExposureKey = m.ExposureKey
WHERE m.SoMSourceBucket > 0
AND m.SoMDenouncedStatus <> 'WRITE OFF'
'''
)
AssignmentDataMapping_v1.createOrReplaceTempView("AssignmentDataMapping_v1")

##### AFMTotalEBAHubsRiskBucket

In [0]:
AFMTotalEBAHubsRiskBucket = spark.sql(
f'''
SELECT Entity
	,   CASE WHEN PE1_3_count > 0	THEN 'PE 1-3'
			 WHEN PE0_count > 0	THEN 'PE 0'
			 WHEN NPE_Count > 0	THEN 'NPE'
		END PE0PE1_3NPEDebtor
FROM (
SELECT 
        Entity
    ,   SUM(CASE WHEN PE0PE1_3NPEAccount = 'PE 1-3'  THEN 1 ELSE 0 END) AS  PE1_3_count
	,   SUM(CASE WHEN PE0PE1_3NPEAccount = 'PE 0'	  THEN 1 ELSE 0 END) AS PE0_count
	,   SUM(CASE WHEN PE0PE1_3NPEAccount = 'NPE'	  THEN 1 ELSE 0 END) AS NPE_count
FROM AssignmentDataMapping_v1
WHERE AssignmentChannel ilike 'Hub RMs%' 
GROUP BY Entity) a

'''
)
AFMTotalEBAHubsRiskBucket.createOrReplaceTempView("AFMTotalEBAHubsRiskBucket")

##### AFMTotalEBAHubsHighBalance

In [0]:
AFMTotalEBAHubsHighBalance = spark.sql(
f'''
SELECT Entity
	, CASE WHEN HighBalance_Count > 0 			THEN 'High Balance'
		   WHEN PE_count > 0 AND NPE_Count > 0 THEN 'PE/NPE'
		   WHEN PE_count > 0 AND NPE_Count = 0 THEN 'PE'
		   WHEN PE_count = 0 AND NPE_Count > 0 THEN 'NPE'
		END AS PENPEHighBalanceDebtor
FROM (
SELECT 
        Entity
    ,   SUM(CASE WHEN AssignmentHub = '2130'  THEN 1 ELSE 0 END) AS HighBalance_Count
	,   SUM(CASE WHEN PENPEAccount = 'PE'	  THEN 1 ELSE 0 END) AS PE_count
	,   SUM(CASE WHEN PENPEAccount = 'NPE'	  THEN 1 ELSE 0 END) AS NPE_count
FROM AssignmentDataMapping_v1
WHERE AssignmentChannel ilike 'Hub RMs%' 
GROUP BY Entity) a

'''
)
AFMTotalEBAHubsHighBalance.createOrReplaceTempView("AFMTotalEBAHubsHighBalance")

##### AFMTotalEBA

In [0]:
AFMTotalEBA = spark.sql(
f'''
SELECT Entity
	, CASE WHEN PE_count > 0 AND NPE_Count > 0 THEN 'PE/NPE'
		   WHEN PE_count > 0 AND NPE_Count = 0 THEN 'PE'
		   WHEN PE_count = 0 AND NPE_Count > 0 THEN 'NPE'
		END AS PENPEDebtor
FROM (
SELECT 
        Entity
	,   SUM(CASE WHEN PENPEAccount = 'PE'	  THEN 1 ELSE 0 END) AS PE_count
	,   SUM(CASE WHEN PENPEAccount = 'NPE'	  THEN 1 ELSE 0 END) AS NPE_count
FROM AssignmentDataMapping_v1
GROUP BY Entity) a

'''
)
AFMTotalEBA.createOrReplaceTempView("AFMTotalEBA")

##### AFMTotalHubEBA

In [0]:
AFMTotalHubEBA = spark.sql(
f'''
SELECT Entity
	, CASE WHEN PE_count > 0 AND NPE_Count > 0 THEN 'PE/NPE'
		   WHEN PE_count > 0 AND NPE_Count = 0 THEN 'PE'
		   WHEN PE_count = 0 AND NPE_Count > 0 THEN 'NPE'
		END AS PENPEHubDebtor
FROM (
SELECT 
        Entity
	,   SUM(CASE WHEN PENPEAccount = 'PE'	  THEN 1 ELSE 0 END) AS PE_count
	,   SUM(CASE WHEN PENPEAccount = 'NPE'	  THEN 1 ELSE 0 END) AS NPE_count
FROM AssignmentDataMapping_v1
WHERE AssignmentChannel ilike 'Hub RMs%' 
GROUP BY Entity) a

'''
)
AFMTotalHubEBA.createOrReplaceTempView("AFMTotalHubEBA")

##### fasttrack

In [0]:
fasttrack = spark.sql(
f'''
SELECT DISTINCT
		    ExposureKey
    ,		PositiveForInstallmentsExtensionReduction
	,		SubmitDate
FROM `{env_prefix}-cdh`.externalfiles.extfasttrack
WHERE PositiveForInstallmentsExtensionReduction = '486  -  ΘΕΤΙΚΟΣ_LOW_INST_NO_PJ'
AND SubmitDate <= {run_date!r} 
AND SubmitDate > {som_date!r}
'''
)
fasttrack.createOrReplaceTempView("fasttrack")

#####  maxinflows

In [0]:
maxinflows = spark.sql(
f'''
SELECT  ExposureNumber
    ,   TargetDate
    ,   InflowMonth
FROM `{env_prefix}-cdh-delivery`.repoint.core_maxinflows
WHERE BusinessDate = {som_date!r}
'''
)
maxinflows.createOrReplaceTempView("maxinflows")

##### lpfx364

In [0]:
lpfx364 = spark.sql(
f'''
SELECT DISTINCT
		    OldExposurekey
    ,		Forbearance
	,		CASE 
        -- If the value matches the format 'YYYY-MM-DD HH:mm:ss.0'
        WHEN SUBSTRING(RestructuringDate, 5, 1) = '-' THEN try_cast(RestructuringDate AS date)
        -- If the value matches the format 'DD/MM/YYYY'
        WHEN SUBSTRING(RestructuringDate, 3, 1) = '/' THEN TO_DATE(RestructuringDate, 'dd/MM/yyyy')
        -- For any other format, assuming it's already in the desired format
    END AS RestructuringDate
FROM `{env_prefix}-cdh`.externalfiles.extlpfxb364
WHERE BusinessDate = {run_date!r}
'''
)
lpfx364.createOrReplaceTempView("lpfx364")

# Check if the dataset is empty and print a message accordingly
if lpfx364.count() == 0:
    print(f"The required external file extlpfxb364 dataset for dates {run_date!r} is empty")

##### modifications

In [0]:
modifications = spark.sql(
f'''
SELECT * 
FROM (
SELECT DISTINCT
		  M.ExposureKey
  ,		M.ImplementationDate
	,		M.Segment
	,		M.ModificationProduct AS ProductType
	,		L.Forbearance
	,		RANK() OVER(PARTITION BY M.ExposureKey ORDER BY M.ImplementationDate ASC) AS Rank 
FROM pmmodifications M
LEFT JOIN lpfx364 L ON M.ExposureKey = L.OldExposureKey
WHERE M.ImplementationDate IS NOT NULL) a
WHERE a.Rank = 1
'''
)
modifications.createOrReplaceTempView("modifications")

##### modifications_hist

In [0]:
modifications_hist = spark.sql(
f'''
SELECT DISTINCT
		     M.OldExposureKey	AS ExposureKey
    ,		M.RestructuringDate AS  ImplementationDate
FROM `{env_prefix}-cdh`.reference.corestatichistoricalmodificationbeforemigration M
WHERE last_day(M.RestructuringDate) <= {soq_date!r}
UNION 
SELECT DISTINCT
		    M.OldExposureKey	AS ExposureKey
    ,		M.RestructuringDate AS  ImplementationDate
FROM lpfx364 M
WHERE last_day(M.RestructuringDate) <= {soq_date!r}
'''
)
modifications_hist.createOrReplaceTempView("modifications_hist")

##### reportunified_soq_v1

In [0]:
reportunified_soq_v1 = spark.sql(
f'''
SELECT * 
FROM (
SELECT * 
,   RANK() OVER (PARTITION BY ExposureKey ORDER BY CurrentStepDate DESC, FirstDateReceiptValuation DESC) AS Rank
FROM reportunified u
WHERE ApplicationNumber <> ''
AND UnifiedApplicationStatus = 'Ενεργή'
--AND FirstDateReceiptValuation IS NOT NULL
AND BusinessDate = {soq_date!r}) u 
WHERE u.Rank = 1
'''
)
reportunified_soq_v1.createOrReplaceTempView("reportunified_soq_v1")

##### reportunified_som_v1

In [0]:
reportunified_som_v1 = spark.sql(
f'''
SELECT * 
FROM (
SELECT * 
,   RANK() OVER (PARTITION BY ExposureKey ORDER BY CurrentStepDate DESC, FirstDateReceiptValuation DESC) AS Rank
FROM reportunified u
WHERE ApplicationNumber <> ''
AND UnifiedApplicationStatus = 'Ενεργή'
AND FirstDateReceiptValuation IS NOT NULL
AND BusinessDate = {som_date!r}) u 
WHERE u.Rank = 1
'''
)
reportunified_som_v1.createOrReplaceTempView("reportunified_som_v1")

##### reportunified_cur_v1

In [0]:
reportunified_cur_v1 = spark.sql(
f'''
SELECT * 
FROM reportunified u
WHERE ApplicationNumber <> ''
AND BusinessDate = {run_date!r}
'''
)
reportunified_cur_v1.createOrReplaceTempView("reportunified_cur_v1")

##### reportunified_soq

In [0]:
reportunified_soq = spark.sql(
f'''
SELECT DISTINCT u.* 
FROM reportunified_soq_v1 u 
LEFT JOIN (
SELECT DISTINCT U.ApplicationNumber, u.ExposureKey
FROM reportunified_soq_v1 u
INNER JOIN modifications_hist m ON u.ExposureKey = m.ExposureKey
WHERE m.ImplementationDate <= {soq_date!r}
AND m.ImplementationDate >= u.FirstDateReceiptValuation) m 
ON u.ApplicationNumber = m.ApplicationNumber AND u.ExposureKey = m.ExposureKey
WHERE m.ApplicationNumber IS NULL AND m.Exposurekey IS NULL
'''
)
reportunified_soq.createOrReplaceTempView("reportunified_soq")

##### reportunified_som

In [0]:
reportunified_som = spark.sql(
f'''
SELECT DISTINCT u.* 
FROM reportunified_som_v1 u 
LEFT JOIN (
SELECT DISTINCT U.ApplicationNumber, u.ExposureKey
FROM reportunified_som_v1 u
INNER JOIN modifications_hist m ON u.ExposureKey = m.ExposureKey
WHERE m.ImplementationDate <= {soq_date!r}
AND m.ImplementationDate >= u.FirstDateReceiptValuation) m 
ON u.ApplicationNumber = m.ApplicationNumber AND u.ExposureKey = m.Exposurekey
WHERE m.Exposurekey IS NULL AND m.ApplicationNumber IS NULL
'''
)
reportunified_som.createOrReplaceTempView("reportunified_som")

##### reportunified_cur_v2

In [0]:
reportunified_cur_v2 = spark.sql(
f'''
SELECT DISTINCT u.* 
FROM reportunified_cur_v1 u 
LEFT JOIN (
SELECT DISTINCT U.ApplicationNumber, u.ExposureKey
FROM reportunified_cur_v1 u
INNER JOIN modifications_hist m ON u.ExposureKey = m.ExposureKey
WHERE m.ImplementationDate <= {soq_date!r}
AND m.ImplementationDate >= u.FirstDateReceiptValuation
) m 
ON u.ApplicationNumber = m.ApplicationNumber AND u.ExposureKey = m.ExposureKey
LEFT JOIN reportunified_soq u_soq ON u.ApplicationNumber = u_soq.ApplicationNumber
WHERE m.Exposurekey IS NULL 
AND m.ApplicationNumber IS NULL 
AND (u_soq.ApplicationNumber IS NOT NULL OR u.ApplicationInsertDate > {soq_date!r} OR u.FirstDateReceiptValuation > {soq_date!r})
'''
)
reportunified_cur_v2.createOrReplaceTempView("reportunified_cur_v2")

##### reportunified_cur

In [0]:
reportunified_cur = spark.sql(
f'''
SELECT  u.* 
 ,		CASE WHEN ApplicationActivationBranch <> '0374' AND Hub IS NULL AND Channel IS NULL THEN CONCAT('ΚΑΤΑΣΤΗΜΑ ',ApplicationActivationBranch) 
			 WHEN ApplicationActivationBranch = '0374' AND Hub LIKE '%ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ%' THEN Hub
			 WHEN Hub = 'CALL CENTER - ALPHA BANK ΥΠΗΡΕΣΙΕΣ ΕΝΗΜΕΡΩΣΕΩΣ' OR Channel = 'ALPHA BANK ΥΠΗΡΕΣΙΕΣ ΕΝΗΜΕΡΩΣΕΩΣ ΟΦΕΙΛΕΤΩΝ' THEN 'DNC'
			 WHEN (Hub NOT LIKE '%ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-%' OR Hub <> 'CALL CENTER - ALPHA BANK ΥΠΗΡΕΣΙΕΣ ΕΝΗΜΕΡΩΣΕΩΣ') AND Channel <> 'ΠΕΡΙΦΕΡΕΙΑΚΟ ΚΕΝΤΡΟ' THEN CONCAT('ΣΥΝΕΡΓΑΤΕΣ ',Channel)
			 WHEN (Hub NOT LIKE '%ΠΕΡ. ΚΕΝΤΡΟ ΔΙΑΧ/ΣΕΩΣ ΚΑΘ/ΣΕΩΝ-%' OR Hub <> 'CALL CENTER - ALPHA BANK ΥΠΗΡΕΣΙΕΣ ΕΝΗΜΕΡΩΣΕΩΣ') AND Channel = 'ΠΕΡΙΦΕΡΕΙΑΚΟ ΚΕΝΤΡΟ' THEN CONCAT('ΣΥΝΕΡΓΑΤΕΣ ', 'ΠΕΡΙΦΕΡΕΙΑΚΟ ΚΕΝΤΡΟ')
		END ApplicationChannelCategory
,	step.Category AS StepCategory
,	m.ImplementationDate
FROM (
SELECT u.* 
, RANK() OVER(PARTITION BY U.ExposureKey ORDER BY	CASE WHEN u.UnifiedApplicationStatus = 'Ενεργή' THEN 1 ELSE 0 END DESC,
																					  u.CurrentStepDate DESC, u.FirstDateReceiptValuation DESC) AS Rank
FROM reportunified_cur_v2 u) u 
LEFT JOIN `{env_prefix}-reportinputfiles`.repoint.core_staticapplicationstepmapping AS step  ON step.CurrentStep = u.CurrentApplicationStep
LEFT JOIN pmmodifications m	    ON u.ExposureKey = m.ExposureKey
WHERE Rank = 1
'''
)
reportunified_cur.createOrReplaceTempView("reportunified_cur")

##### AgentInfo for Activityuser and ActivityUserID

In [0]:
AgentInfo = spark.sql(
f'''
select distinct EmployeePartyKey as CreatedBy, EmployeeFullName as PartyName,
                EmployeeUserName as EmpLogin, rank() OVER(PARTITION BY EmployeeFullName order by TeamCode desc) AS rank    
from `{env_prefix}-cdh`.primary.firmstructure fs
WHERE 1=1
AND EmployeeCategory = "Employee"
and SourceSystem = 'qca'
and fs.BusinessDate = {run_date!r}
''')   
AgentInfo.createOrReplaceTempView("AgentInfo")

##### ExternalModifications

In [0]:
ExternalModifications = spark.sql(
f'''
select distinct em.Portfolio, em.CIF, em.AFM, em.EBAStatus, em.AccountNumber, cast(em.ApplicationFlag as int) ApplicationFlag, 
em.FinalSaved, em.SavedReason, em.ImplementationDate, em.AttributionChannel, em.AttributionTeam, em.AttributionAgent, case when lower(em.SavedDate) in('null','') then null else cast(em.SavedDate as date) end as SavedDate, cast(em.SomBalance as decimal(19,2)) as SomBalance, em.ExtraDPOFlag   
from `{env_prefix}-reportinputfiles`.repoint.core_externalfileattributions em
WHERE last_day(em.ReferenceDateEOM) = last_day({run_date!r})
''')   
ExternalModifications.createOrReplaceTempView("ExternalModifications")

##### modificationsproduction

In [0]:
modificationsproduction = spark.sql(
f'''
SELECT 
	    {run_date!r} AS ReferenceDate
	,	CASE WHEN em.CIF IS NOT NULL THEN em.CIF
 			 ELSE a.CDI END AS CDI
	,	CASE WHEN em.AFM IS NOT NULL THEN em.AFM
 			 ELSE a.Entity END AS PartyTaxRegistrationNumber
	,	c.FullName AS CustomerName
	,	ul.CalcLawStatus AS L3869Status
	,	a.AgreementKey
	,	a.ExposureKey 
	,	a.AssetSegment
	,	a.Treatmentbranch AS TreatmentBranch
	,	a.BRANCH_DESC AS Branch
	,	CASE WHEN em.SomBalance IS NOT NULL THEN em.SomBalance
 			 ELSE a.SoMBalanceAmountEUR end AS SoMBalance
	,	a.SoMSourceBucket AS SoMBucket
	,	a.SoMDenouncedStatus  
	,	a.RiskBucket
	,	CASE WHEN em.EBAStatus IS NOT NULL THEN em.EBAStatus
 			 ELSE a.EBAStatus end AS EBAStatus
 	,	CASE WHEN em.EBAStatus IN ('NPL','FNPL')		THEN 'NPE'
  			 WHEN pm.NPEFinalizedInflow = 1 THEN 'NPE Inflow' ELSE a.PE0PE1_3NPEAccount END AllFinalEBAStatus
 	,	CASE WHEN em.EBAStatus IN ('NPL','FNPL')		THEN 'NPE'
			 WHEN em.EBAStatus IN ('PL','FPL','CFPL')  THEN 'PE'
	         ELSE a.PENPEAccount 
		END AS PENPEAccount
	,	a.PE0PE1_3NPEAccount
	,	t.`PENPEDebtor`
	,	CASE WHEN a.AssignmentChannel ilike 'Hub RMs%' THEN h.`PENPEHubDebtor` END PENPEHubDebtor
	,	a.CourseOfAction
	,	CASE WHEN sb.Entity IS NOT NULL THEN 'SBP' ELSE 'Retail' END AS CustomerRetailCategory
	,	a.AssignmentChannel AS AssignmentChannel
	,	a.AssignmentTeam AS AssignmentChannelDetailed
	,	a.AssignmentAgent
	,	ag1.EmpLogin AS AssignmentAgentAM
	,	m.Segment AS AccountSegment
	,	a.AssignmentSegmentDetailed
	,	CASE WHEN em.FinalSaved IS NOT NULL THEN em.FinalSaved
 			 ELSE si.FinalSaved end AS FinalSavedFlag
	,	CASE WHEN em.SavedReason IS NOT NULL THEN em.SavedReason
 			 ELSE si.SavedReason END AS FinalSavedReason
	,	CASE WHEN em.ImplementationDate IS NOT NULL THEN em.ImplementationDate
 			 ELSE m.ImplementationDate END AS ImplementationDate
	,	CASE WHEN em.ExtraDPOFlag = 'YES' THEN 'DPO'
 			 ELSE m.ProductType END AS ProductType
	,	m.Forbearance
 	,	ft.SubmitDate AS LowInstallmentDate
	,	f.TargetDate TargetDate
	,	f.InflowMonth InflowMonth
	,	CASE WHEN uf_soq.FirstDateReceiptValuation <= {soq_date!r} THEN 1 END ActivePipelineSoQ
	,	CASE WHEN uf_som.FirstDateReceiptValuation <= {som_date!r} THEN 1 END ActivePipelineSoM
	,	CASE WHEN em.ApplicationFlag = 1 THEN 1
 			 WHEN uf.FirstDateReceiptValuation > {soq_date!r} THEN 1 END NewPipelineWithinQuarter
	,	CASE WHEN em.ApplicationFlag = 1 THEN 1
 			 WHEN uf.FirstDateReceiptValuation > {som_date!r} THEN 1 END NewPipelineWithinMonth
	,	CASE WHEN em.AccountNumber IS NOT NULL AND em.SavedReason <> 'Payments' THEN 1
 			 WHEN m.ImplementationDate > {soq_date!r} AND m.ImplementationDate <= {run_date!r} THEN 1 END ImplementedWithinQuarter
	,	CASE WHEN em.AccountNumber IS NOT NULL AND em.SavedReason <> 'Payments' THEN 1
 			 WHEN m.ImplementationDate > {som_date!r} AND m.ImplementationDate <= {run_date!r} THEN 1 END ImplementedWithinMonth
	,	CASE WHEN m.ImplementationDate <= {run_date!r} AND m.ProductType not like 'Calypso%' AND (M.ProductType like 'C%' OR  M.ProductType like 'D%') THEN 1 END ImplementedWithHaircutWithinMonth
 	,	uf.ApplicationNumber
	,	uf.ApplicationInsertDate
	,	CASE WHEN uf.ImplementationDate IS NOT NULL				THEN 'Υλοποιήσεις'								 
			 WHEN uf.UnifiedApplicationStatus = 'Ολοκληρωμένη' 	THEN 'Ακυρωμένη / ολοκληρωμένη χωρίς υλοποίηση' 
			 ELSE uf.StepCategory 
		END AS StepCategory
	,	uf.CurrentApplicationStep
	,	CAST(uf.CurrentStepDate AS STRING) ApplicationCurrentStepInsertDate
	,	CAST(uf.FirstDateReceiptValuation AS STRING) FirstDateAtValidation
	,	uf.ApplicationChannelCategory
	,	uf.UnifiedApplicationStatus
	,	CASE WHEN uf.ImplementationDate IS NOT NULL					THEN 'Ναι'
			 WHEN uf.Decision in('Έγκριση','Ναι')					THEN 'Ναι'
			 WHEN uf.Decision in('Απόρριψη','Όχι')					THEN 'Όχι'					 
			 WHEN uf.StepCategory LIKE 'ΔΔΟΛΤ%'						THEN NULL
			 WHEN uf.Decision = 'Μη Καταχωρημένο'					THEN NULL					 
		END ApplicationApproval
	,	CASE WHEN uf.ImplementationDate IS NOT NULL					THEN uf.DecisionReason
			 WHEN uf.Decision in('Έγκριση','Ναι')					THEN uf.DecisionReason
			 WHEN uf.StepCategory LIKE 'ΔΔΟΛΤ%'						THEN NULL
			 WHEN uf.Decision IN ('Μη Καταχωρημένο','Απόρριψη')  	THEN NULL					 
		END ApplicationApprovalReason
	,	CASE WHEN uf.NPLSolution = 'Ναι' THEN 1 ELSE 0 END NPLSolution
	,	CASE WHEN uf.DEK = 'Ναι' THEN 1 ELSE 0 END DEK
	,	uf.RestructuringType
	,	uf.HaircutAmount
	,	uf.ReturnReason
	,	uf.ImplementationApplicationNumber
	,	uf.AutoCreditAtApplicationLevel
	,	uf.ReevaluationTimes
	,	uf.RegistratorCode
	,	uf.RegistratorName
	,	uf.ApplicationActivationBranch AS ApplicationBranch
	,	uf.FollowBranch
	,	uf.AutoPrintFlag
	,	uf.PrintException
	,	CASE WHEN TRIM(uf.ApplicationAutoPrintFlag) = '' THEN NULL ELSE uf.ApplicationAutoPrintFlag END AS ApplicationAutoPrintFlag
	,	CASE WHEN TRIM(uf.ApplicationPreAppovedFlag) = '' THEN NULL ELSE uf.ApplicationPreAppovedFlag END AS ApplicationPreAppovedFlag
	,	uf.ApplicationAutocredit
	,	uf.InitialSignature
	,	uf.ApprovalDifferentTerms
	,	uf.ApplicationApprovalWithDifferentTerms
	,	uf.ApplicationSignatureRequired
	,	CASE WHEN TRIM(uf.SignedByBankDate) = '' THEN NULL ELSE uf.SignedByBankDate END AS SignedByBankDate
	,	uf.L4605Flag
	,	uf.Covid19
	,	uf.Authority
	,	CASE WHEN a.FinalAssignmentChannel  = 'IRT' AND a.AssignmentSegmentDetailed ILIKE '%DPO%'				THEN 'DPO'
			 WHEN a.FinalAssignmentChannel  = 'IRT' AND a.AssignmentSegmentDetailed ILIKE '%pjextension%'		THEN 'pjextension'
			 WHEN a.FinalAssignmentChannel  = 'IRT' AND a.AssignmentSegmentDetailed ILIKE '%lowinstallment%'	THEN 'lowinstallment'
			 WHEN a.FinalAssignmentChannel  = 'IRT' AND a.AssignmentSegmentDetailed ILIKE '%lowinstallment%'	THEN 'lowinstallment'
			 WHEN a.FinalAssignmentChannel  = 'IRT'																THEN pm.EBAIndicator
		END AS DNCcategory																					
	,	CASE WHEN em.ApplicationFlag = 1 AND EM.AttributionChannel ILIKE '%IRT%' THEN 1
 				 WHEN a.FinalAssignmentChannel  = 'IRT' AND uf_som.ExposureKey IS  NULL AND IFNULL(uf.FirstDateReceiptValuation,'1900-01-01') > {som_date!r} THEN 1 
			 WHEN a.FinalAssignmentChannel  = 'IRT' AND (ia_486.ExposureKey IS NOT NULL OR ia_620.ExposureKey IS NOT NULL)							  THEN 1
		END NewPipelineWithinMonth_IRT  
	,	CASE WHEN EM.AttributionChannel ILIKE '%IRT%' AND em.AccountNumber IS NOT NULL AND em.SavedReason <> 'Payments' THEN 1
 				 WHEN a.FinalAssignmentChannel  = 'IRT' AND m.ImplementationDate > {som_date!r} AND m.ImplementationDate <= {run_date!r}	THEN 1 END ImplementedWithinMonth_IRT  
	,	CASE WHEN a.FinalAssignmentChannel  = 'IRT' THEN la.InteractionTypeKey END ActivityTypeKey_IRT
	,	CASE WHEN a.FinalAssignmentChannel  = 'IRT' THEN la.InteractionType END ActivityName_IRT
	,	CASE WHEN a.FinalAssignmentChannel  = 'IRT' THEN la.InteractionDate END ActivityDate_IRT
	,	CASE WHEN a.FinalAssignmentChannel  = 'IRT' THEN ag.PartyName END ActivityUser_IRT
	,	CASE WHEN a.FinalAssignmentChannel  = 'IRT' THEN ag.Emplogin END ActivityUserId_IRT
	,	CASE WHEN em.AccountNumber IS NOT NULL THEN em.AttributionChannel
 			 ELSE a.FinalAssignmentChannel END AS FinalAssignmentChannel
	,	CASE WHEN em.AccountNumber IS NOT NULL THEN em.AttributionTeam
 			 ELSE a.FinalAssignmentTeam END AS FinalAssignmentTeam
	,	CASE WHEN em.AccountNumber IS NOT NULL THEN aiem.PartyName
 			 ELSE a.FinalAssignmentAgent end as FinalAssignmentAgent
	,	CASE WHEN em.AccountNumber IS NOT NULL THEN em.AttributionAgent
 			 ELSE ifnull(a.FinalAssignmentAgentAM, ag1.EmpLogin) end AS FinalAssignmentAgentAM
 	,	CASE WHEN em.AccountNumber IS NOT NULL THEN em.AttributionChannel
 			 ELSE a.AttributionChannel END AS AttributionChannel
	,	CASE WHEN em.AccountNumber IS NOT NULL THEN em.AttributionTeam
 			 ELSE a.AttributionTeam END AS AttributionTeam
  	,	CASE WHEN em.AccountNumber IS NOT NULL THEN aiem.PartyName
 			 ELSE a.AttributionAgent END AS AttributionAgent
	,	CASE WHEN em.AccountNumber IS NOT NULL THEN em.AttributionAgent
 			 ELSE a.AttributionAgentAM END AS AttributionAgentAM
 	,	CASE WHEN em.AccountNumber IS NOT NULL AND em.SavedReason = 'Payments' THEN em.AttributionChannel
 			 ELSE a.PaymentAttributionChannel END AS PaymentAttributionChannel
	,	CASE WHEN em.AccountNumber IS NOT NULL AND em.SavedReason = 'Payments' THEN em.AttributionTeam
 			 ELSE a.PaymentAttributionTeam END AS PaymentAttributionTeam
  	,	CASE WHEN em.AccountNumber IS NOT NULL AND em.SavedReason = 'Payments' THEN aiem.PartyName
 			 ELSE a.PaymentAttributionAgent END AS PaymentAttributionAgent
	,	CASE WHEN em.AccountNumber IS NOT NULL AND em.SavedReason = 'Payments' THEN em.AttributionAgent
 			 ELSE a.PaymentAttributionAgentAM END AS PaymentAttributionAgentAM
	,	a.PaymentAttributionAmount
	,	CASE WHEN em.AccountNumber IS NOT NULL THEN 1 ELSE 0 END AS ExternalFileSource
FROM AssignmentDataMapping_v1 a
INNER JOIN AFMTotalEBA t on a.Entity = t.Entity
LEFT JOIN AFMTotalHubEBA h on a.Entity = h.Entity
LEFT JOIN (SELECT DISTINCT Entity FROM AssignmentDataMapping_v1 WHERE AssetSegment = 'SB') sb on a.Entity = sb.Entity
LEFT JOIN modifications m on a.ExposureKey = m.ExposureKey
LEFT JOIN dparty c on a.CDI = c.CDI
LEFT JOIN dunderlaw ul on a.CDI = ul.CDI
LEFT JOIN reportunified_cur uf on uf.ExposureKey = a.ExposureKey
LEFT JOIN reportunified_som uf_som on uf_som.ExposureKey = a.ExposureKey and uf_som.ApplicationNumber = uf.ApplicationNumber
LEFT JOIN reportunified_soq uf_soq on uf_soq.ExposureKey = a.ExposureKey and uf_soq.ApplicationNumber = uf.ApplicationNumber
LEFT JOIN savedinfo si on si.AccountNumber = a.ExposureKey
LEFT JOIN pmmodifications pm on a.ExposureKey = pm.ExposureKey
LEFT JOIN (SELECT DISTINCT ExposureKey FROM interaction WHERE InteractionTypeKey = '486') ia_486 on a.ExposureKey = ia_486.ExposureKey
LEFT JOIN (SELECT DISTINCT ExposureKey FROM interaction WHERE InteractionTypeKey = '620') ia_620 on a.ExposureKey = ia_620.ExposureKey
LEFT JOIN (SELECT DISTINCT ExposureKey, InteractionTypeKey, InteractionType, InteractionDate, CreatedBy, CreatedById 
		   FROM interaction WHERE Rank = 1) la on a.ExposureKey = la.ExposureKey
LEFT JOIN fasttrack ft on a.ExposureKey = ft.ExposureKey
LEFT JOIN maxinflows f on a.ExposureKey = f.ExposureNumber
left join AgentInfo ag on la.createdby = ag.CreatedBy 
left join AgentInfo ag1 on a.AssignmentAgent = ag1.PartyName  and ag1.Rank = 1
LEFT JOIN ExternalModifications em on em.AccountNumber = a.ExposureKey
LEFT JOIN AgentInfo AIEM on em.AttributionAgent = aiem.EmpLogin

UNION ALL 

SELECT 
	    {run_date!r} AS ReferenceDate
	,	em.CIF AS CDI
	,	em.AFM AS PartyTaxRegistrationNumber
	,	NULL AS CustomerName
	,	NULL L3869Status
	,	NULL AS AgreementKey
	,	em.AccountNumber AS ExposureKey
	,	NULL AS AssetSegment
	,	NULL AS TreatmentBranch
	,	NULL AS Branch
	,	em.SomBalance
	,	NULL AS SoMBucket
	,	NULL AS SoMDenouncedStatus  
	,	NULL AS RiskBucket
	,	em.EBAStatus
 	,	CASE WHEN em.EBAStatus IN ('NPL','FNPL')		THEN 'NPE' END AS AllFinalEBAStatus
	,	CASE WHEN em.EBAStatus IN ('NPL','FNPL')		THEN 'NPE'
				 WHEN em.EBAStatus IN ('PL','FPL','CFPL')  THEN 'PE'
				END AS PENPEAccount
	,	NULL AS PE0PE1_3NPEAccount
	,	NULL AS `PENPEDebtor`
	,	NULL AS PENPEHubDebtor
	,	NULL AS CourseOfAction
	,	NULL AS CustomerRetailCategory
	,	NULL AS AssignmentChannel
	,	NULL AS AssignmentChannelDetailed
	,	NULL AS AssignmentAgent
	,	NULL AS AssignmentAgentAM
	,	NULL AS AccountSegment
	,	NULL AS AssignmentSegmentDetailed
	,	em.FinalSaved AS FinalSavedFlag
	,	em.SavedReason AS FinalSavedReason
	,	em.ImplementationDate
	,	CASE WHEN em.ExtraDPOFlag = 'YES' THEN 'DPO'    
 			  END AS ProductType
	,	NULL AS Forbearance
 	,	NULL AS LowInstallmentDate
	,	NULL AS TargetDate
	,	NULL AS InflowMonth
	,	NULL AS ActivePipelineSoQ
	,	NULL AS ActivePipelineSoM
	,	CASE WHEN em.ApplicationFlag = 1 THEN 1 END NewPipelineWithinQuarter
	,	CASE WHEN em.ApplicationFlag = 1 THEN 1 END NewPipelineWithinMonth
	,	CASE WHEN em.AccountNumber IS NOT NULL AND em.SavedReason <> 'Payments' THEN 1
 			 END ImplementedWithinQuarter
	,	CASE WHEN em.AccountNumber IS NOT NULL AND em.SavedReason <> 'Payments' THEN 1
 			 END ImplementedWithinMonth
	,	NULL ImplementedWithHaircutWithinMonth
 	,	NULL AS ApplicationNumber
	,	NULL AS ApplicationInsertDate
	,	NULL AS StepCategory
	,	NULL AS CurrentApplicationStep
	,	NULL AS ApplicationCurrentStepInsertDate
	,	NULL AS FirstDateAtValidation
	,	NULL AS ApplicationChannelCategory
	,	NULL AS UnifiedApplicationStatus
	,	NULL AS ApplicationApproval
	,	NULL AS ApplicationApprovalReason
	,	NULL AS NPLSolution
	,	NULL AS DEK
	,	NULL AS RestructuringType
	,	NULL AS HaircutAmount
	,	NULL AS ReturnReason
	,	NULL AS ImplementationApplicationNumber
	,	NULL AS AutoCreditAtApplicationLevel
	,	NULL AS ReevaluationTimes
	,	NULL AS RegistratorCode
	,	NULL AS RegistratorName
	,	NULL AS ApplicationBranch
	,	NULL AS FollowBranch
	,	NULL AS AutoPrintFlag
	,	NULL AS PrintException
	,	NULL AS ApplicationAutoPrintFlag
	,	NULL AS ApplicationPreAppovedFlag
	,	NULL AS ApplicationAutocredit
	,	NULL AS InitialSignature
	,	NULL AS ApprovalDifferentTerms
	,	NULL AS ApplicationApprovalWithDifferentTerms
	,	NULL AS ApplicationSignatureRequired
	,	NULL AS SignedByBankDate
	,	NULL AS L4605Flag
	,	NULL AS Covid19
	,	NULL AS Authority
	,	NULL AS DNCcategory																					
	,	CASE WHEN em.ApplicationFlag = 1 AND EM.AttributionChannel ILIKE '%IRT%' THEN 1 
		END NewPipelineWithinMonth_IRT
	,	CASE WHEN EM.AttributionChannel ILIKE '%IRT%' AND em.AccountNumber IS NOT NULL AND em.SavedReason <> 'Payments' THEN 1
 	 	END ImplementedWithinMonth_IRT
	,	NULL AS ActivityTypeKey_IRT
	,	NULL AS ActivityName_IRT
	,	NULL AS ActivityDate_IRT
	, NULL AS ActivityUser_IRT
	, NULL AS ActivityUserId_IRT
	,	em.AttributionChannel AS FinalAssignmentChannel
	,	em.AttributionTeam AS FinalAssignmentTeam
	,	aiem.PartyName as FinalAssignmentAgent
	,	em.AttributionAgent AS FinalAssignmentAgentAM
 	,	CASE WHEN em.AccountNumber IS NOT NULL THEN em.AttributionChannel
 			END AS AttributionChannel
	,	CASE WHEN em.AccountNumber IS NOT NULL THEN em.AttributionTeam
 			END AS AttributionTeam
  	,	CASE WHEN em.AccountNumber IS NOT NULL THEN aiem.PartyName
 			END AS AttributionAgent
	,	CASE WHEN em.AccountNumber IS NOT NULL THEN em.AttributionAgent
 			END AS AttributionAgentAM
 	,	CASE WHEN em.AccountNumber IS NOT NULL AND em.SavedReason = 'Payments' THEN em.AttributionChannel
 			END AS PaymentAttributionChannel
	,	CASE WHEN em.AccountNumber IS NOT NULL AND em.SavedReason = 'Payments' THEN em.AttributionTeam
 			END AS PaymentAttributionTeam
  	,	CASE WHEN em.AccountNumber IS NOT NULL AND em.SavedReason = 'Payments' THEN aiem.PartyName
 			END AS PaymentAttributionAgent
	,	CASE WHEN em.AccountNumber IS NOT NULL AND em.SavedReason = 'Payments' THEN em.AttributionAgent
 			END AS PaymentAttributionAgentAM
	,	NULL AS PaymentAttributionAmount
	,	CASE WHEN em.AccountNumber IS NOT NULL THEN 1 ELSE 0 END AS ExternalFileSource
FROM ExternalModifications em
LEFT JOIN AssignmentDataMapping a on em.AccountNumber = a.ExposureKey
LEFT JOIN AgentInfo AIEM on em.AttributionAgent = aiem.EmpLogin
WHERE a.ExposureKey is null

'''
)
modificationsproduction.createOrReplaceTempView("modificationsproduction")

In [0]:
final_df = spark.sql(
f'''
SELECT DISTINCT 
    CAST(ReferenceDate AS DATE) AS BusinessDate,
    CAST(CDI AS STRING) AS CDI,
    CAST(PartyTaxRegistrationNumber AS STRING) AS PartyTaxRegistrationNumber,
    CAST(CustomerName AS STRING) AS CustomerName,
    CAST(L3869Status AS STRING) AS L3869Status,
    CAST(AgreementKey AS STRING) AS AgreementKey,
    CAST(ExposureKey AS STRING) AS ExposureKey,
    CAST(AssetSegment AS STRING) AS AssetSegment,
    CAST(TreatmentBranch AS STRING) AS TreatmentBranch,
    CAST(Branch AS STRING) AS Branch,
    CAST(SoMBalance AS DECIMAL(19,2)) AS SoMBalance,
    CAST(SoMBucket AS INT) AS SoMBucket,
    CAST(SoMDenouncedStatus AS STRING) AS SoMDenouncedStatus,
    CAST(RiskBucket AS INT) AS RiskBucket,
    CAST(EBAStatus AS STRING) AS EBAStatus,
    CAST(AllFinalEBAStatus AS STRING) AS AllFinalEBAStatus,
    CAST(PENPEAccount AS STRING) AS PENPEAccount,
    CAST(PE0PE1_3NPEAccount AS STRING) AS PE0PE1_3NPEAccount,
    CAST(PENPEDebtor AS STRING) AS PENPEDebtor,
    CAST(PENPEHubDebtor AS STRING) AS PENPEHubDebtor,
    CAST(CourseOfAction AS STRING) AS CourseOfAction,
    CAST(CustomerRetailCategory AS STRING) AS CustomerRetailCategory,
    CAST(AssignmentChannel AS STRING) AS AssignmentChannel,
    CAST(AssignmentChannelDetailed AS STRING) AS AssignmentChannelDetailed,
    CAST(AssignmentAgent AS STRING) AS AssignmentAgent,
    CAST(AssignmentAgentAM AS STRING) AS AssignmentAgentAM,
    CAST(AccountSegment AS STRING) AS AccountSegment,
    CAST(AssignmentSegmentDetailed AS STRING) AS AssignmentSegmentDetailed,
    CAST(FinalSavedFlag AS STRING) AS FinalSavedFlag,
    CAST(FinalSavedReason AS STRING) AS FinalSavedReason,
    CAST(ImplementationDate AS DATE) AS ImplementationDate,
    CAST(ProductType AS STRING) AS ProductType,
    CAST(Forbearance AS STRING) AS Forbearance,
    CAST(LowInstallmentDate AS DATE) AS LowInstallmentDate,
    CAST(TargetDate AS DATE) AS TargetDate,
    CAST(InflowMonth AS STRING) AS InflowMonth,
    CAST(ActivePipelineSoQ AS INT) AS ActivePipelineSoQ,
    CAST(ActivePipelineSoM AS INT) AS ActivePipelineSoM,
    CAST(NewPipelineWithinQuarter AS INT) AS NewPipelineWithinQuarter,
    CAST(NewPipelineWithinMonth AS INT) AS NewPipelineWithinMonth,
    CAST(ImplementedWithinQuarter AS INT) AS ImplementedWithinQuarter,
    CAST(ImplementedWithinMonth AS INT) AS ImplementedWithinMonth,
    CAST(ImplementedWithHaircutWithinMonth AS INT) AS ImplementedWithHaircutWithinMonth,
    CAST(ApplicationNumber AS STRING) AS ApplicationNumber,
    CAST(ApplicationInsertDate AS DATE) AS ApplicationInsertDate,
    CAST(StepCategory AS STRING) AS StepCategory,
    CAST(CurrentApplicationStep AS STRING) AS ApplicationCurrentStep,
    CAST(ApplicationCurrentStepInsertDate AS DATE) AS ApplicationCurrentStepInsertDate,
    CAST(FirstDateAtValidation AS DATE) AS FirstDateAtValidation,
    CAST(ApplicationChannelCategory AS STRING) AS ApplicationChannelCategory,
    CAST(UnifiedApplicationStatus AS STRING) AS ApplicationStatus,
    CAST(ApplicationApproval AS STRING) AS ApplicationApproval,
    CAST(ApplicationApprovalReason AS STRING) AS ApplicationApprovalReason,
    CAST(NPLSolution AS INT) AS NPLSolution,
    CAST(DEK AS INT) AS DEK,
    CAST(RestructuringType AS STRING) AS ModificationType,
    CAST(HaircutAmount AS DECIMAL(19,2)) AS HaircutAmount,
    CAST(ReturnReason AS STRING) AS ReturnReason,
    CAST(ImplementationApplicationNumber AS STRING) AS ImplementationApplicationNumber,
    CAST(AutoCreditAtApplicationLevel AS INT) AS ApplicationLevelAutoCredit,
    CAST(ReevaluationTimes AS INT) AS ReevaluationTimes,
    CAST(RegistratorCode AS STRING) AS ApplicationRegistratorAM,
    CAST(RegistratorName AS STRING) AS ApplicationRegistratorName,
    CAST(ApplicationBranch AS STRING) AS ApplicationBranch,
    CAST(FollowBranch AS STRING) AS FollowBranch,
    CAST(AutoPrintFlag AS STRING) AS AutomaticPrinting,
    CAST(PrintException AS STRING) AS PrintException,
    CAST(ApplicationAutoPrintFlag AS INT) AS IndicatorOfAutomatedPrintingAtApplicationLevel,
    CAST(ApplicationPreAppovedFlag AS INT) AS IndicatorPreapprovedAtApplicationLevel,
    CAST(ApplicationAutocredit AS STRING) AS ApplicationAutocredit,
    CAST(InitialSignature AS STRING) AS InitialSignature,
    CAST(ApprovalDifferentTerms AS STRING) AS ApprovalDifferentTerms,
    CAST(ApplicationApprovalWithDifferentTerms AS STRING) AS ApplicationApprovalWithDifferentTerms,
    CAST(ApplicationSignatureRequired AS STRING) AS ApplicationSignatureRequired,
    CAST(SignedByBankDate AS DATE) AS SignedByBankDate,
    CAST(L4605Flag AS STRING) AS L4605Flag,
    CAST(Covid19 AS STRING) AS Covid19Flag,
    CAST(Authority AS STRING) AS DiscretionOfApproval,
    CAST(DNCcategory AS STRING) AS DNCcategory,
    CAST(NewPipelineWithinMonth_IRT AS INT) AS NewPipelineWithinMonth_IRT,
    CAST(ImplementedWithinMonth_IRT AS INT) AS ImplementedWithinMonth_IRT,
    CAST(ActivityTypeKey_IRT AS STRING) AS ActivityTypeKey_IRT,
    CAST(ActivityName_IRT AS STRING) AS ActivityName_IRT,
    CAST(ActivityDate_IRT AS DATE) AS ActivityDate_IRT,
    CAST(ActivityUser_IRT AS STRING) AS ActivityUser_IRT,
    CAST(ActivityUserId_IRT AS STRING) AS ActivityUserId_IRT,
    CAST(FinalAssignmentChannel AS STRING) AS FinalAssignmentChannel,
    CAST(FinalAssignmentTeam AS STRING) AS FinalAssignmentTeam,
    CAST(FinalAssignmentAgent AS STRING) AS FinalAssignmentAgent,
    CAST(FinalAssignmentAgentAM AS STRING) AS FinalAssignmentAgentAM,
    CAST(AttributionChannel AS STRING) AS AttributionChannel,
    CAST(AttributionTeam AS STRING) AS AttributionTeam,
    CAST(AttributionAgent AS STRING) AS AttributionAgent,
    CAST(AttributionAgentAM AS STRING) AS AttributionAgentAM,
    CAST(PaymentAttributionChannel AS STRING) AS PaymentAttributionChannel,
    CAST(PaymentAttributionTeam AS STRING) AS PaymentAttributionTeam,
    CAST(PaymentAttributionAgent AS STRING) AS PaymentAttributionAgent,
    CAST(PaymentAttributionAgentAM AS STRING) AS PaymentAttributionAgentAM,
    CAST(PaymentAttributionAmount AS DECIMAL(19,2)) AS PaymentAttributionAmount,
    CAST(ExternalFileSource AS INT) AS ExternalFileSource,
    CAST(getdate() AS TIMESTAMP) AS InsertTimeStamp
FROM modificationsproduction
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_ModificationsProduction: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
ModificationsProduction = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
ModificationsProduction.createOrReplaceTempView("ModificationsProduction")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/ModificationsProduction/"
file_name = "ModificationsProduction_" +run_date+ ".csv"
ModificationsProduction.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)